# Sucking problem Description

Same [testcase](http://basilisk.fr/sandbox/ecipriano/run/suckingproblem.c) can be find in the following work:

Edoardo Cipriano, Abd Essamade Saufi, Alessio Frassoldati, Tiziano Faravelli, Stéphane Popinet, and Alberto Cuoci. _Multicomponent Droplet Evaporation in a Geometric Volume-Of-Fluid Framework_. Journal of Computational Physics, 507:112955, June 2024



## Objective
This (quasi) 1D test case is widely used for the verification of phase-change models, which involves phase change driven by temperature gradients.

## Remark
In the sucking problem, a temperature gradient within the **liquid** region induces evaporation, which in turn drives the interface velocity. The governing equations are solved on the **liquid side only**.

## Summary of Initial and Boundary Conditions
- **Boundary Conditions (BCs):**
  - **Dynamics**: 
    - Free-slip condition on the top and bottom walls.
    - Fixed wall on the left side.
    - Outlet condition on the right side.
  - **Thermal**: 
    - Adiabatic conditions on the top and bottom walls.
    - Fixed temperature at $ T_w $ on the right wall.
    - Fixed temperature at $ T_{\text{sat}} $ on the left boundary.
  - **Phase**: All boundaries set to symmetry.

- **Initial Conditions (ICs):**
  - **Dynamics**: Initial velocity is zero throughout the domain.
  - **Thermal**: The analytical solution of temperature is impose within the liquid, ranging from $ T_{\text{sat}} $ at $ x=x_{i,int} $ to  $ T_w $ at $ x=L$. The vapor is initially at a uniform temperature of $ T_{\text{sat}} $.
  - **Phase**: Vapor extends from $ x=0 $ to $ x=x_{i,int} $.

![ICs and BCs for Stefan Problem](src/figs/sucking.png)

## Case Specifications
- **Domain Length**: $ L = 10 \, \text{mm} $
- **Initial Interface Position**: $ x_i = 0.476 \, \text{mm} $ at $ t = 0 \, \text{s} $
- **Material Properties**: Properties for water under atmospheric conditions:

|               | $\rho$ $kg/m^3$| $\mu$ $Pa\cdot s$        | $\lambda$ $W/(m\cdot K)$ | $C_p$ $J/(kg\cdot K)$   |
|---------------|----------------|--------------------------|--------------------------|-------------------------|
| **Liquid**    | 958.37         | $2.8 \times 10^{-4}$     | 0.679                    | $4.21 \times 10^3$      |
| **Vapor**     | 0.597          | $1.227 \times 10^{-5}$   | 0.025                    | $2.077 \times 10^3$     |
|               |                |                          |                          |                         |
| $\sigma = 0.0589$ N/m, $\mathcal{L} = 2.256 \times 10^6$ J/kg                                                  | 

---

## Analytical Solution

The analytical solution provides the theoretical interface position $ x_i(t) $ and the temperature distribution $ \Delta T(x, t) $ within the liquid (!Attention in the equation, it's  $\alpha_v$).

$$
x_i(t) = 2 \beta \sqrt{\alpha_v t}
$$

$$
T = \Delta T - \left( \frac{\Delta T}{\text{erfc}\left( \frac{\beta \cdot \rho_v \cdot \sqrt{\alpha_v}}{\rho_l \cdot \sqrt{\alpha_l}} \right)} \right) \cdot \text{erfc}\left( \frac{x}{2 \cdot \sqrt{\alpha_l \cdot (t + t_{\text{shift}})}} + \frac{\beta \cdot (\rho_v - \rho_l)}{\rho_l}\cdot \sqrt{\frac{\alpha_v}{\alpha_l}} \right)
$$

where:
$\alpha_v$ is the thermal diffusivity of the vapor, defined by $$\alpha_v=\frac{\lambda_v}{\rho_v cp_{v}} $$; 

$cp_v$ is the specific heat capacity of the vapor (J/kg·K); $\operatorname{erf}(\beta)$ is the error function; 


$\beta$ is determined by solving the transcendental equation:

$$
f(\beta, \Delta T) = \exp(\beta^2) \cdot \text{erf}(\beta) \cdot \left( \beta - \frac{\Delta T \cdot C_{p_v} \cdot \lambda_l \cdot \sqrt{\alpha_v} \cdot \exp\left(- \left( \frac{\beta \cdot \rho_v \cdot \sqrt{\alpha_v}}{\rho_l \cdot \sqrt{\alpha_l}} \right)^2 \right)}{h_{lg} \cdot \lambda_v \cdot \sqrt{\pi \cdot \alpha_l} \cdot \text{erfc}\left( \frac{\beta \cdot \rho_v \cdot \sqrt{\alpha_v}}{\rho_l \cdot \sqrt{\alpha_l}} \right)} \right)
$$


## Anlytical solution and pre-load module

In [2]:
import numpy as np
from scipy import special
from math import sqrt, pi
from scipy.optimize import fsolve
from scipy.optimize import brentq
import matplotlib.pyplot as plt

Mar  = ['o', 's', '*', 'D','H']
colors = ['red', 'blue', 'black', 'green', 'orange', 'purple']

try:
    import medcoupling as mc  # Replace 'some_module' with the name of the module you want to load
    print("medcoupling loaded successfully!")
except ImportError as e:
    print("Failed to load module:", e)
    try:
        import scienceplots
        plt.style.use('science')
    except ImportError as e:
        print("Failed to load module:", e)
import os
current_directory = os.getcwd() 


# Constants
Cp_v = 2.077e3   # Specific heat capacity of vapor (J/kg·K)
lambda_v = 0.025  # Thermal conductivity of vapor (W/m·K)
rho_v = 0.597     # Density of vapor (kg/m^3)
alpha_v = lambda_v / (rho_v * Cp_v)  # Thermal diffusivity of vapor (m^2/s)

# Liquid properties
rho_l = 958.37    # Density of liquid (kg/m^3)
lambda_l = 0.679  # Thermal conductivity of liquid (W/m·K)
Cp_l = 4.21e3     # Specific heat capacity of liquid (J/kg·K)
alpha_l = lambda_l / (rho_l * Cp_l)  # Thermal diffusivity of liquid (m^2/s)

h_lg = 2.256e6  # Latent heat of vaporization (J/kg)
dT = 5.0       # Temperature difference (K)

# Define the function for Brent's method, which requires the function to have opposite signs at the bounds
def sol_sucking(beta, dT):
    coef = np.exp(beta**2) * special.erf(beta)
    coef_exp = beta * rho_v * sqrt(alpha_v) / (rho_l * sqrt(alpha_l))
    
    return coef * (beta - dT * Cp_v * lambda_l * sqrt(alpha_v) *
                   np.exp(-coef_exp**2) / (h_lg * lambda_v * sqrt(pi * alpha_l) * 
                   special.erfc(coef_exp)))

Failed to load module: No module named 'medcoupling'


In [5]:
def find_root(dT):
    # Brent's method requires an interval where the function changes sign
    beta_min = 0.1  # Lower bound for beta
    beta_max = 5.0  # Upper bound for beta

    # Use Brent's method to find the root of sol_sucking
    root = brentq(sol_sucking, beta_min, beta_max, args=(dT,))
    
    return root

# Solve the equation for the given temperature difference
beta = find_root(dT)

# Output the solution
print(f"The root of the equation is beta = {beta:.5f}")


The root of the equation is beta = 0.77669


In [6]:
xi_0= 0.476e-3
tshift = (xi_0/2./beta)**2/alpha_v
def x_position(t):
    return 2.0 * beta * np.sqrt(alpha_v * (t+tshift))
def temp(t, x):
    term1 = dT
    term2 = (dT / special.erfc(beta * rho_v * sqrt(alpha_v) / (rho_l * sqrt(alpha_l))))
    term3 = special.erfc(x / (2 * sqrt(alpha_l * (t + tshift))) + beta * (rho_v - rho_l) / (rho_l )* sqrt(alpha_v / alpha_l))
    # Return the temperature as the sum of the terms
    return term1 - term2 * term3

# Run set

In [8]:
from trustutils import run

run.introduction("L. WEI","09/10/2024")
# Declaration of the TRUST version
run.TRUST_parameters("1.9.3")

## Introduction 
 Validation made by : L. WEI



 Report created : 09/10/2024



 Report generated 21/11/2024

### TRUST parameters 
 * Version TRUST: 1.9.3
 * Binary used: /volatile/catC/linkai/TRIOCFD/TrioCFD_opt (built in directory /volatile/catC/linkai/TRIOCFD/share/Validation/Rapports_automatiques/Multiphase/Front_tracking_discontinu/sucking/build)

In [9]:
from math import sqrt, pi, floor, log10

# dt_max = [dt_popinet(N-1) for N in nombre_de_noeuds ]

nombre_de_noeuds = [65, 129, 257, 513]
nb_test = len(nombre_de_noeuds)
dt_max = [1.e-1]*nb_test

a = rho_v * sqrt(alpha_v) / (rho_l * sqrt(alpha_l))
b  = (rho_v - rho_l) / (rho_l )* sqrt(alpha_v / alpha_l)

cond_init = f'{dT}*(1-(1.-erf((x/{2.*sqrt(alpha_l*tshift):.3e})+({b*beta:.3e})))/(1.-erf({beta*a:.3e})))'

case_test = 'sucking'
run.reset()

nbprocs = 1
for y in range(nb_test) :
# for y in range(1) :
    fname =f'{case_test}_{str(nombre_de_noeuds[y]-1)}'
    name = f'{case_test}_{str(nombre_de_noeuds[y]-1)}'
    substitutions_dict = {
                          "nbn" : str(nombre_de_noeuds[y]),
                          "mdtmax" :str(dt_max[y]), 
                          "delatT" :str(dT),
                           "beta" :cond_init
                          }

    tc = run.addCaseFromTemplate("sucking.data"
                             ,targetDirectory=f"{fname}"
                             ,dic=substitutions_dict
                             ,nbProcs= nbprocs
                             ,targetData=f"{name}.data")



    if nbprocs > 1:
        tc.partition()
run.printCases()


### Test cases 
* sucking_64/sucking_64.data 
* sucking_128/sucking_128.data 
* sucking_256/sucking_256.data 
* sucking_512/sucking_512.data 


In [ ]:
# Run all cases
run.runCases()
# run.runCases(verbose=False, preventConcurrent=True)

In [10]:
## Performances calculs

run.tablePerf()

RuntimeError: 
Execution of following command failed!!
  /volatile/catC/linkai/trust_dev/Validation/Outils/Genere_courbe/scripts/extract_perf sucking_256
with return code 1
and with following output:

grep: sucking_256.err: No such file or directory
-> Calculation sucking_256 produced an error !
-> Look at the /volatile/catC/linkai/TRIOCFD/share/Validation/Rapports_automatiques/Multiphase/Front_tracking_discontinu/sucking/build/sucking_256/ file.
grep: sucking_256.TU: No such file or directory
grep: sucking_256.TU: No such file or directory
grep: sucking_256.err: No such file or directory


# Funcitons for postprocessing

In [12]:
def get_prb_data(file, time):
    from trustutils.files import SonSEGFile
 
    import numpy as np
    
    
    son_file = f'build/{file}'
    donne = SonSEGFile(son_file,None)
    compo = 0
    ncompo = donne.getnCompo()
    entries = donne.getEntries()
    
    # y_label = entries[compo].split()[0]
    # x_label = donne.getXLabel()
    
    # print("x_label : ", x_label)
    # print("y_label : ", y_label)
    
    # start, end = donne.getXTremePoints()
    # print("dom xmin and ymin : ", start)
    # print("dom xmax and ymax : ", end)
    
    t = donne.getValues(entries[0])[0]

    ## find closest value of t
    if time == None:
        idx = -1
    else:
        idx = (np.abs(t - time)).argmin()
        
    X = donne.getXAxis()
    
    Y = []
    for i in entries[compo::ncompo]:
        Y.append(list(donne.getValues(i)[1])[idx])
    if X[0] != X.min():
        X = X[::-1]
        Y = Y[::-1]
    return X, np.asarray(Y)
def get_position(fname, name, W):
    import numpy as np
    import os
    os.system(f'grep "^Volume_phase_0" build/{fname}/{name}.err > build/{fname}/vol.txt')
     
    time = []
    vol = []  
    with open(f'build/{fname}/vol.txt', 'r') as file:
    # Lire chaque ligne du fichier
        for line in file:
            # Séparer les valeurs de la ligne en utilisant l'espace comme délimiteur
            valeurs = line.split()
            # Ajouter la valeur de la deuxième colonne à la liste colonne_2
            vol.append(float(valeurs[1]))
            # Ajouter la valeur de la quatrième colonne à la liste colonne_4
            time.append(float(valeurs[3]))
        
    posi = np.array(vol) / W
    return np.asarray(time), posi

# Temperature along x

In [14]:
L = 0.01
tmax = 0.2
W = 0.002

In [16]:
num_mar = 0
for y in range(nb_test) :
# for y in range(1) :
    fname =f'{case_test}_{str(nombre_de_noeuds[y]-1)}'
    name =f'{case_test}_{str(nombre_de_noeuds[y]-1)}'
    X, Y = get_prb_data(f'{fname}/{name}_T_ABSCISSE.son', tmax)
    freq = int(len(X)/80)+1
    plt.scatter(X[::freq]/xi_0,Y[::freq]/dT
                ,marker=Mar[num_mar%len(Mar)], s=30, facecolors='none', edgecolors='k'
         ,  label=r'$\Delta x=$' +f'{2**(-y)*xi_0*1.e6:.0f}'+ r'$\mathrm{\mu m}$'
        )
    num_mar = num_mar +1

y = 0
x = np.linspace(0, L, 101)

T_ana = temp(tmax, x)
plt.plot(x/xi_0,T_ana/dT, 'b'
         ,  label='Ana.'
        )
num_mar = num_mar +1 

    
plt.xlabel(r'$x/x_{i}$')
plt.ylabel(r"$(T-T_\mathrm{sat})/\Delta T$")                  
plt.legend(loc="best")
# plt.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.4))
# plt.xlim(0, 1)
plt.ylim(0, 1)
plt.show()

IndexError: list index out of range

# Interface position

In [ ]:
# Main plot
zoom_view = False
fig, ax = plt.subplots()
num_mar = 0
for y in range(nb_test) :   
    fname =f'{case_test}_{str(nombre_de_noeuds[y]-1)}'
    name = f'{case_test}_{str(nombre_de_noeuds[y]-1)}'
    time, position = get_position(fname,name, W)
    # freq = int(len(time)/20)+2*y
    freq =int(len(time)/20) +1
    ax.scatter(time[::freq]/tshift+1,position[::freq]/xi_0
             ,marker=Mar[num_mar%len(Mar)], s=30, facecolors='none', edgecolors='k'
         ,  label=r'$\Delta x=$' +f'{2**(-y)*xi_0*1.e6:.0f}'+ r'$\mathrm{\mu m}$'
        )
    num_mar = num_mar +1
y= 0
time = np.linspace(0, tmax, 101)
x_ana = x_position(time)
plt.plot(time/tshift+1,x_ana/xi_0
         , color=colors[(num_mar//2)%len(colors)]
         ,  label='Ana.'
        )

num_mar = num_mar +1

ax.set_xlabel(r'$t/t_{\mathrm{ref}}$')
ax.set_ylabel(r"$x_i/x_{i, int}$")                  
ax.legend(loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.3))
plt.show()



# Relative error at the end of simu 

In [ ]:
err = []
for y in range(nb_test) :
    fname =f'{case_test}_{str(nombre_de_noeuds[y]-1)}'
    with open(f'build/{fname}/vol.txt', 'r') as file:
        last_line = file.readlines()[-1].strip()
        valeurs = last_line.split()
        vol = float(valeurs[1])
        time= float(valeurs[3])
        x_simu = np.asarray(vol)/W
        x_theric = x_position(time)
        err.append(abs(x_simu-x_theric)/x_theric)


x = np.asarray(nombre_de_noeuds) -1 
plt.scatter(x, err,marker=Mar[num_mar%len(Mar)], s=30, facecolors='none', edgecolors='k', label = f'present')

plt.plot(x, 10000/x**2, label='2nd order')  
plt.plot(x, 10/x, label='1st order')

plt.xlabel(r"Number of cells")
plt.ylabel(r"Relative error")    
plt.xscale('log', base=2)
plt.yscale('log')
# plt.ylim([1.e-2, 1])
plt.legend(loc="best")
plt.show()
